[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/opherdonchin/ModelsOfTheMotorSystems/blob/master/In%20class%20exercises/Ex6_Fitts_Law.ipynb)


# Lecture 6 In-Class Assignment: Hick's Law and Fitts' Law

This exercise analyzes two classic regularities in motor behavior:
1. **Hick's Law**: reaction time increases as the number of possible choices increases.
2. **Fitts' Law**: pointing movement time increases as targets become smaller or farther away.

For both laws, you will compare your own data to reference data from Opher.

**Goal**: Fit linear models for both laws and compare your fitted constants to Opher's constants.

## Outline
1. [Imports & Setup](#section1)
2. [Hick's Law Demo Data](#section2)
3. [Process Hick's Law Data](#section3)
4. [Fit Opher's Hick's Law Data](#section4)
5. [Fit Your Hick's Law Data](#section5)
6. [Compare Hick's Law Constants](#section6)
7. [Load Fitts' Law Data](#section7)
8. [Preprocess Fitts' Law Trials](#section8)
9. [Fit Opher's Fitts' Law Data](#section9)
10. [Fit Your Fitts' Law Data](#section10)
11. [Compare Fitts' Law Constants](#section11)
12. [Discussion](#section12)

<a id="section1"></a>
## 1) Imports & Setup

**DO NOT EDIT**: Basic imports, plotting defaults, file names, pasted demo data, and helper functions.

In [ ]:
import csv
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

In [ ]:
plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

In [ ]:
REFERENCE_DATA_FILE = "opherdonchin_trackpad_fitts_law_results_1779164739097.csv"
REFERENCE_DATA_URL = (
    "https://raw.githubusercontent.com/opherdonchin/ModelsOfTheMotorSystems/master/"
    "In%20class%20exercises/opherdonchin_trackpad_fitts_law_results_1779164739097.csv"
)
STUDENT_DATA_FILE = "my_fitts_law_results.csv"  # Replace with your downloaded Fitts' Law CSV filename.

HICKS_REFERENCE_DATA = """
1:                              0.274
2:                              1.631
5:                              1.885
8:                              3.807
8:                              2.876
9:                              2.158
5:                              1.869
9:                              2.556
9:                              2.034
10:                            2.003
"""

HICKS_STUDENT_DATA = """
"""  # Paste your copied Hick's Law demo data here.

In [ ]:
def _as_float(value):
    if value is None or value == "":
        return np.nan
    try:
        return float(value)
    except (TypeError, ValueError):
        return np.nan


def _as_int(value):
    if value is None or value == "":
        return None
    try:
        return int(float(value))
    except (TypeError, ValueError):
        return None


def _as_bool(value):
    if isinstance(value, bool):
        return value
    if value is None or value == "":
        return True
    text = str(value).strip().lower()
    if text in {"true", "1", "yes", "y", "hit", "success", "successful"}:
        return True
    if text in {"false", "0", "no", "n", "miss", "failure", "failed"}:
        return False
    return bool(value)


def parse_hicks_data(pasted_text):
    """Parse copied Hick's Law demo data into cleaned trial dictionaries."""
    trials = []
    for line in pasted_text.strip().splitlines():
        if ":" not in line:
            continue
        choices_text, time_text = line.split(":", 1)
        choices = _as_float(choices_text.strip())
        reaction_time = _as_float(time_text.strip())
        if not np.all(np.isfinite([choices, reaction_time])):
            continue
        if choices <= 0 or reaction_time <= 0:
            continue
        trials.append(
            {
                "choices": int(choices),
                "bits": np.log2(choices),
                "reaction_time": reaction_time,
            }
        )
    return trials


def print_hicks_data(trials):
    print(f"{'choices':>7} {'bits':>8} {'RT (s)':>8}")
    for trial in trials:
        print(f"{trial['choices']:>7} {trial['bits']:>8.3f} {trial['reaction_time']:>8.3f}")


def print_hicks_fit(label, fit):
    print(label)
    print(f"  n trials:   {fit['n']}")
    print(f"  intercept:  {fit['intercept']:.3f} s")
    print(f"  slope:      {fit['slope']:.3f} s / bit")
    print(f"  R squared:  {fit['r_squared']:.3f}")


def print_hicks_fit_comparison(comparison):
    print("Comparison to Opher's Hick's Law reference fit")
    print("  intercept")
    print(f"    student:   {comparison['student_intercept']:.3f} s")
    print(f"    reference: {comparison['reference_intercept']:.3f} s")
    print(
        f"    difference: {comparison['intercept_difference']:.3f} s "
        f"({comparison['intercept_percent_difference']:.1f}%)"
    )
    print("  slope")
    print(f"    student:   {comparison['student_slope']:.3f} s / bit")
    print(f"    reference: {comparison['reference_slope']:.3f} s / bit")
    print(
        f"    difference: {comparison['slope_difference']:.3f} s / bit "
        f"({comparison['slope_percent_difference']:.1f}%)"
    )


def plot_hicks_fit(trials, fit, label="Demo data", color="tab:blue"):
    if len(trials) == 0:
        print(f"No Hick's Law trials to plot for {label}.")
        return

    bits = np.array([trial["bits"] for trial in trials])
    reaction_times = np.array([trial["reaction_time"] for trial in trials])

    plt.scatter(bits, reaction_times, alpha=0.75, label=label, color=color)
    if np.isfinite(fit["slope"]):
        x_line = np.linspace(bits.min(), bits.max(), 100)
        y_line = fit["intercept"] + fit["slope"] * x_line
        plt.plot(x_line, y_line, color=color)

    plt.xlabel("Information, log2(number of choices)")
    plt.ylabel("Reaction time (s)")
    plt.legend()
    plt.tight_layout()


def plot_hicks_fit_comparison(student_trials, student_fit, reference_trials, reference_fit):
    plt.figure(figsize=(7, 4))
    plot_hicks_fit(reference_trials, reference_fit, label="Opher reference", color="tab:orange")
    plot_hicks_fit(student_trials, student_fit, label="Student", color="tab:blue")
    plt.title("Hick's Law fits")
    plt.show()


def plot_hicks_constant_differences(comparison):
    labels = ["intercept", "slope"]
    differences = [comparison["intercept_difference"], comparison["slope_difference"]]

    plt.figure(figsize=(5, 3.5))
    plt.axhline(0, color="0.3", linewidth=1)
    plt.bar(labels, differences, color=["tab:blue", "tab:orange"])
    plt.ylabel("Student - Opher")
    plt.title("Difference in Hick's Law constants")
    plt.tight_layout()
    plt.show()


def load_fitts_csv(csv_path, label, fallback_url=None):
    """Load a Fitts' Law CSV exported from the class demo."""
    path = Path(csv_path)
    if path.exists():
        with path.open(newline="", encoding="utf-8-sig") as csv_file:
            rows = list(csv.DictReader(csv_file))
        print(f"Loaded {len(rows)} rows for {label} from {path}")
        return rows

    if fallback_url is not None:
        try:
            from urllib.request import urlopen

            with urlopen(fallback_url) as response:
                text = response.read().decode("utf-8-sig")
            rows = list(csv.DictReader(text.splitlines()))
            print(f"Loaded {len(rows)} rows for {label} from the online reference file")
            return rows
        except Exception as error:
            print(f"Could not load the online reference file for {label}: {error}")

    print(f"Could not find {path} for {label}.")
    return []


def preview_trials(trials, n=5, max_columns=8):
    """Print a compact preview of the first few trial dictionaries."""
    for row in trials[:n]:
        items = list(row.items())[:max_columns]
        print(dict(items))


def prepare_trials(raw_trials, dataset_label):
    """Keep valid hit trials and standardize the exported columns."""
    clean_trials = []
    for row in raw_trials:
        hit = _as_bool(row.get("hit", True))
        movement_time = _as_float(row.get("time"))
        index_of_difficulty = _as_float(row.get("id"))
        distance = _as_float(row.get("distanceFromPrevious"))
        radius = _as_float(row.get("radius"))
        misses = _as_int(row.get("misses"))

        if not hit:
            continue
        if not np.all(np.isfinite([movement_time, index_of_difficulty])):
            continue
        if movement_time <= 0:
            continue

        clean_trials.append(
            {
                "dataset": dataset_label,
                "username": row.get("username", dataset_label),
                "device": row.get("device", ""),
                "sequence": _as_int(row.get("sequence")),
                "rep": _as_int(row.get("rep")),
                "distance": distance,
                "radius": radius,
                "ID": index_of_difficulty,
                "movement_time": movement_time,
                "IP": _as_float(row.get("ip")),
                "misses": misses,
            }
        )
    return clean_trials


def describe_dataset(trials, label):
    if len(trials) == 0:
        print(f"{label}: no valid trials")
        return

    movement_times = np.array([trial["movement_time"] for trial in trials])
    ids = np.array([trial["ID"] for trial in trials])
    devices = sorted({trial["device"] for trial in trials if trial["device"]})
    users = sorted({trial["username"] for trial in trials if trial["username"]})

    print(f"{label}")
    print(f"  users:   {users}")
    print(f"  devices: {devices}")
    print(f"  trials:  {len(trials)}")
    print(f"  ID range: {ids.min():.2f} to {ids.max():.2f}")
    print(f"  MT range: {movement_times.min():.1f} to {movement_times.max():.1f} ms")


def print_fit(label, fit):
    print(label)
    print(f"  n trials:   {fit['n']}")
    print(f"  intercept:  {fit['intercept']:.2f} ms")
    print(f"  slope:      {fit['slope']:.2f} ms / ID unit")
    print(f"  R squared:  {fit['r_squared']:.3f}")


def print_fit_comparison(comparison):
    print("Comparison to Opher's Fitts' Law reference fit")
    print("  intercept")
    print(f"    student:   {comparison['student_intercept']:.2f} ms")
    print(f"    reference: {comparison['reference_intercept']:.2f} ms")
    print(
        f"    difference: {comparison['intercept_difference']:.2f} ms "
        f"({comparison['intercept_percent_difference']:.1f}%)"
    )
    print("  slope")
    print(f"    student:   {comparison['student_slope']:.2f} ms / ID unit")
    print(f"    reference: {comparison['reference_slope']:.2f} ms / ID unit")
    print(
        f"    difference: {comparison['slope_difference']:.2f} ms / ID unit "
        f"({comparison['slope_percent_difference']:.1f}%)"
    )


def plot_fitts_fit(trials, label, fit, color="tab:blue"):
    if len(trials) == 0:
        print(f"No trials to plot for {label}.")
        return

    ids = np.array([trial["ID"] for trial in trials])
    movement_times = np.array([trial["movement_time"] for trial in trials])

    plt.scatter(ids, movement_times, alpha=0.65, label=label, color=color)
    if np.isfinite(fit["slope"]):
        x_line = np.linspace(ids.min(), ids.max(), 100)
        y_line = fit["intercept"] + fit["slope"] * x_line
        plt.plot(x_line, y_line, color=color)


def plot_fit_comparison(student_trials, student_fit, reference_trials, reference_fit):
    plt.figure(figsize=(7, 4))
    plot_fitts_fit(reference_trials, "Opher reference", reference_fit, color="tab:orange")
    plot_fitts_fit(student_trials, "Student", student_fit, color="tab:blue")
    plt.xlabel("Index of difficulty, ID")
    plt.ylabel("Movement time (ms)")
    plt.title("Fitts' Law fits")
    plt.legend()
    plt.tight_layout()
    plt.show()


def plot_constant_differences(comparison):
    labels = ["intercept", "slope"]
    differences = [comparison["intercept_difference"], comparison["slope_difference"]]

    plt.figure(figsize=(5, 3.5))
    plt.axhline(0, color="0.3", linewidth=1)
    plt.bar(labels, differences, color=["tab:blue", "tab:orange"])
    plt.ylabel("Student - Opher")
    plt.title("Difference in Fitts' Law constants")
    plt.tight_layout()
    plt.show()

<a id="section2"></a>
## 2) Hick's Law Demo Data

Use the Hick's Law demo here: [https://cpe-iitg.vlabs.ac.in/exp/hick-hymans-law/simulation.html](https://cpe-iitg.vlabs.ac.in/exp/hick-hymans-law/simulation.html).

The demo gives copied text with one trial per line. The number before the colon is the number of choices. The number after the colon is reaction time in seconds.

Paste your copied results into `HICKS_STUDENT_DATA` in the setup cell. If `HICKS_STUDENT_DATA` is empty, the notebook uses Opher's reference data as a temporary stand-in so the cells still run.

<a id="section3"></a>
## 3) Process Hick's Law Data

We convert the number of choices into information using:

$$
\text{bits} = \log_2(N)
$$

Then we process Opher's reference data and the student data with the same parser.

In [ ]:
hicks_reference_trials = parse_hicks_data(HICKS_REFERENCE_DATA)
hicks_student_trials = parse_hicks_data(HICKS_STUDENT_DATA)

if len(hicks_student_trials) == 0:
    print("Using Opher's Hick's Law data as a temporary stand-in.")
    print("Paste your copied demo data into HICKS_STUDENT_DATA before doing the comparison.")
    hicks_student_trials = hicks_reference_trials

print("Opher reference")
print_hicks_data(hicks_reference_trials)
print("\nStudent")
print_hicks_data(hicks_student_trials)

<a id="section4"></a>
## 4) Fit Opher's Hick's Law Data

Hick's Law is usually written as:

$$
RT = a + b \log_2(N)
$$

where `a` is a baseline reaction-time cost and `b` is the additional time per bit of choice information.

Complete the function skeleton below. It should take cleaned Hick's Law trials and return the intercept, slope, R squared, and number of trials.

In [ ]:
def fit_hicks_law(trials):
    """Fit RT = a + b * bits and return the fitted constants."""
    # STUDENT CODE HERE
    # Your return value should be a dictionary with these keys:
    # "intercept", "slope", "r_squared", and "n".
    return {
        "intercept": np.nan,
        "slope": np.nan,
        "r_squared": np.nan,
        "n": len(trials),
    }

In [ ]:
hicks_reference_fit = fit_hicks_law(hicks_reference_trials)
print_hicks_fit("Opher Hick's Law reference fit", hicks_reference_fit)

plot_hicks_fit(hicks_reference_trials, hicks_reference_fit, label="Opher reference", color="tab:orange")
plt.title("Opher Hick's Law reference data")
plt.show()

<a id="section5"></a>
## 5) Fit Your Hick's Law Data

Now fit the same Hick's Law model to your own copied demo data.

In [ ]:
hicks_student_fit = fit_hicks_law(hicks_student_trials)
print_hicks_fit("Student Hick's Law fit", hicks_student_fit)

plot_hicks_fit(hicks_student_trials, hicks_student_fit, label="Student", color="tab:blue")
plt.title("Student Hick's Law data")
plt.show()

**Questions**:
- Does your reaction time generally increase with the number of choices?
- Is your fitted line steeper or shallower than Opher's?
- Is your intercept higher or lower than Opher's?

<a id="section6"></a>
## 6) Compare Hick's Law Constants

Now compare your Hick's Law constants to Opher's reference constants.

Complete the comparison function below. It should take two Hick's Law fit dictionaries and compute the raw and percent differences in intercept and slope.

In [ ]:
def compare_hicks_fits(student_fit, reference_fit):
    """Compare a student's Hick's Law constants to Opher's constants."""
    # STUDENT CODE HERE
    # Return the student value, reference value, difference, and percent difference
    # for both intercept and slope.
    return {
        "student_intercept": np.nan,
        "reference_intercept": np.nan,
        "intercept_difference": np.nan,
        "intercept_percent_difference": np.nan,
        "student_slope": np.nan,
        "reference_slope": np.nan,
        "slope_difference": np.nan,
        "slope_percent_difference": np.nan,
    }

In [ ]:
hicks_comparison = compare_hicks_fits(hicks_student_fit, hicks_reference_fit)
print_hicks_fit_comparison(hicks_comparison)

plot_hicks_fit_comparison(hicks_student_trials, hicks_student_fit, hicks_reference_trials, hicks_reference_fit)
plot_hicks_constant_differences(hicks_comparison)

<a id="section7"></a>
## 7) Load Fitts' Law Data

Opher's reference file is already included with this exercise. Use the Fitts' Law demo here: [https://twinji.github.io/fitts-law-input-task/#/](https://twinji.github.io/fitts-law-input-task/#/). Download your own CSV from the demo and place it in the same folder as this notebook.

**Student Task**: Change `STUDENT_DATA_FILE` in the setup cell to your CSV filename.

In [ ]:
reference_raw = load_fitts_csv(REFERENCE_DATA_FILE, "Opher reference", REFERENCE_DATA_URL)
preview_trials(reference_raw)

student_raw = load_fitts_csv(STUDENT_DATA_FILE, "student")

if len(student_raw) == 0:
    print("Using Opher's file as a temporary stand-in so the notebook can run.")
    print("Replace STUDENT_DATA_FILE with your own CSV before doing the comparison.")
    student_raw = reference_raw

<a id="section8"></a>
## 8) Preprocess Fitts' Law Trials

We use the demo's exported `id` column directly. This avoids mixing different Fitts' Law conventions. We keep successful trials and standardize the column names used by the plotting and fitting functions.

**Student Task**: Create `reference_trials` and `student_trials` using `prepare_trials`.

In [ ]:
# STUDENT CODE HERE
reference_trials = prepare_trials(reference_raw, "Opher reference")
student_trials = prepare_trials(student_raw, "student")

describe_dataset(reference_trials, "Opher reference")
describe_dataset(student_trials, "Student")

<a id="section9"></a>
## 9) Fit Opher's Fitts' Law Data

First fit Opher's Fitts' Law reference dataset so there is a fixed comparison point for everyone.

Before fitting either dataset, complete the function skeleton below. This is the only Fitts' Law analysis function you need to write: it should take a list of cleaned trials and return the intercept, slope, R squared, and number of trials.

In [ ]:
def fit_fitts_law(trials):
    """Fit MT = a + b * ID and return the fitted constants."""
    # STUDENT CODE HERE
    # Your return value should be a dictionary with these keys:
    # "intercept", "slope", "r_squared", and "n".
    return {
        "intercept": np.nan,
        "slope": np.nan,
        "r_squared": np.nan,
        "n": len(trials),
    }

In [ ]:
reference_fit = fit_fitts_law(reference_trials)
print_fit("Opher Fitts' Law reference fit", reference_fit)

plot_fitts_fit(reference_trials, "Opher reference", reference_fit, color="tab:orange")
plt.xlabel("Index of difficulty, ID")
plt.ylabel("Movement time (ms)")
plt.title("Opher Fitts' Law reference data")
plt.legend()
plt.tight_layout()
plt.show()

<a id="section10"></a>
## 10) Fit Your Fitts' Law Data

Now fit the same Fitts' Law model to your own data.

In [ ]:
# STUDENT CODE HERE
student_fit = fit_fitts_law(student_trials)
print_fit("Student Fitts' Law fit", student_fit)

plot_fitts_fit(student_trials, "Student", student_fit, color="tab:blue")
plt.xlabel("Index of difficulty, ID")
plt.ylabel("Movement time (ms)")
plt.title("Student Fitts' Law data")
plt.legend()
plt.tight_layout()
plt.show()

**Questions**:
- Does your movement time increase with `ID`?
- Is your fitted line steeper or shallower than Opher's?
- Is your intercept higher or lower than Opher's?

<a id="section11"></a>
## 11) Compare Fitts' Law Constants

The intercept `a` is a baseline movement-time cost. The slope `b` is the extra movement time per unit increase in index of difficulty.

**Student Task**: Compare your fitted constants to Opher's reference constants.

Now complete the Fitts' Law comparison function. It should take the two fit dictionaries and compute the raw and percent differences in intercept and slope.

In [ ]:
def compare_fits(student_fit, reference_fit):
    """Compare a student's fitted constants to Opher's reference constants."""
    # STUDENT CODE HERE
    # Your return value should include the student value, reference value,
    # difference, and percent difference for both intercept and slope.
    return {
        "student_intercept": np.nan,
        "reference_intercept": np.nan,
        "intercept_difference": np.nan,
        "intercept_percent_difference": np.nan,
        "student_slope": np.nan,
        "reference_slope": np.nan,
        "slope_difference": np.nan,
        "slope_percent_difference": np.nan,
    }

In [ ]:
comparison = compare_fits(student_fit, reference_fit)
print_fit_comparison(comparison)

plot_fit_comparison(student_trials, student_fit, reference_trials, reference_fit)
plot_constant_differences(comparison)

<a id="section12"></a>
## 12) Discussion

Please edit this markdown cell to answer:

1. How different is your Hick's Law intercept from Opher's? What might explain that difference?
2. How different is your Hick's Law slope from Opher's? What might explain that difference?
3. How different is your Fitts' Law intercept from Opher's? What might explain that difference?
4. How different is your Fitts' Law slope from Opher's? What might explain that difference?
5. Which model describes your data better: Hick's Law or Fitts' Law? Use the plots and R squared values to support your answer.